In [ ]:
!pip install pymupdf pdfplumber pytesseract pillow
!pip install sentence-transformers faiss-cpu
!pip install groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.7 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^

KeyboardInterrupt: 

In [ ]:
!apt-get install tesseract-ocr -y

In [ ]:
import fitz  # PyMuPDF
import pdfplumber
import pytesseract
from PIL import Image
import os

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from groq import Groq

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
pdf_path = list(uploaded.keys())[0]
pdf_path

In [ ]:
def extract_text_from_pdf(pdf_path):
    text = ""

    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        page = doc[page_num]
        page_text = page.get_text()

        # If text extraction fails → use OCR
        if not page_text.strip():
            pix = page.get_pixmap()
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            page_text = pytesseract.image_to_string(img)

        text += page_text + "\n"

    return text

In [ ]:
raw_text = extract_text_from_pdf(pdf_path)
print(raw_text[:1000])  # preview

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
chunks = chunk_text(raw_text)
len(chunks)

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
chunk_embeddings = model.encode(chunks)

In [ ]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings))

In [ ]:
def retrieve_chunks(query, k=3):
    query_embedding = model.encode([query])

    distances, indices = index.search(np.array(query_embedding), k)

    results = [chunks[i] for i in indices[0]]

    return results

In [ ]:
retrieve_chunks("What is a data incident?")

In [ ]:
import os
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
def ask_question(query):
    context = retrieve_chunks(query)

    combined_context = "\n\n".join(context)

    prompt = f"""
    Answer the question using the context below.
    If the answer is not in the context, say "Not found".

    Context:
    {combined_context}

    Question:
    {query}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
ask_question("Summarize this document")

In [ ]:
def summarize_document():
    summary_prompt = f"""
    Provide a concise summary of the following document:

    {raw_text[:4000]}
    """

    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": summary_prompt}]
    )

    return response.choices[0].message.content

In [ ]:
len(chunks)

In [ ]:
for i in range(3):
    print(f"\n--- Chunk {i} ---\n")
    print(chunks[i][:500])

In [ ]:
retrieve_chunks("What is the revenue mentioned?")

In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('punkt')

In [ ]:
from nltk.tokenize import sent_tokenize

def semantic_chunking(text, chunk_size=500, overlap=2):
    sentences = sent_tokenize(text)

    chunks = []
    current_chunk = []

    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence)

        if current_length + sentence_length > chunk_size:
            chunks.append(" ".join(current_chunk))

            # overlap (last few sentences)
            current_chunk = current_chunk[-overlap:]
            current_length = sum(len(s) for s in current_chunk)

        current_chunk.append(sentence)
        current_length += sentence_length

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [ ]:
import nltk
nltk.download('punkt_tab')

chunks = semantic_chunking(raw_text)
len(chunks)

In [ ]:
chunks = semantic_chunking(raw_text)
len(chunks)

In [ ]:
for i in range(3):
    print(f"\n--- Chunk {i} ---\n")
    print(chunks[i])

In [ ]:
chunk_embeddings = model.encode([c['text'] for c in chunks])

index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
index.add(np.array(chunk_embeddings))

In [ ]:
retrieve_chunks("What is a data incident?")

In [ ]:
def extract_text_with_pages(pdf_path):
    doc = fitz.open(pdf_path)

    pages = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()

        if not text.strip():
            pix = page.get_pixmap()
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(img)

        pages.append({
            "page": page_num,
            "text": text
        })

    return pages

In [ ]:
pages = extract_text_with_pages(pdf_path)
len(pages)

In [ ]:
def semantic_chunking_with_metadata(pages, chunk_size=500, overlap=2):
    from nltk.tokenize import sent_tokenize

    chunks = []

    for page in pages:
        sentences = sent_tokenize(page["text"])

        current_chunk = []
        current_length = 0

        for sentence in sentences:
            if current_length + len(sentence) > chunk_size:
                chunks.append({
                    "text": " ".join(current_chunk),
                    "page": page["page"]
                })

                current_chunk = current_chunk[-overlap:]
                current_length = sum(len(s) for s in current_chunk)

            current_chunk.append(sentence)
            current_length += len(sentence)

        if current_chunk:
            chunks.append({
                "text": " ".join(current_chunk),
                "page": page["page"]
            })

    return chunks

In [ ]:
chunks = semantic_chunking_with_metadata(pages)
len(chunks)

In [ ]:
def retrieve_chunks(query, k=3):
    query_embedding = model.encode([query])

    distances, indices = index.search(np.array(query_embedding), k)

    results = []
    for i in indices[0]:
        results.append({
            "text": chunks[i]["text"],
            "page": chunks[i]["page"]
        })

    return results

In [ ]:
def ask_question(query):
    retrieved = retrieve_chunks(query)

    context = "\n\n".join(
        [f"(Page {r['page']}) {r['text']}" for r in retrieved]
    )

    prompt = f"""
    Answer the question using ONLY the context below.
    Cite page numbers in your answer.
    If not found, say "Not found".

    Context:
    {context}

    Question:
    {query}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
ask_question("What are the five steps to responding to data incidents?")

In [ ]:
import numpy as np

def mmr(query_embedding, doc_embeddings, k=3, lambda_param=0.7):
    selected = []
    candidates = list(range(len(doc_embeddings)))

    similarity = np.dot(doc_embeddings, query_embedding.T).flatten()

    while len(selected) < k and candidates:
        mmr_scores = []

        for idx in candidates:
            relevance = similarity[idx]

            diversity = 0
            if selected:
                diversity = max(
                    np.dot(doc_embeddings[idx], doc_embeddings[s].T)
                    for s in selected
                )

            score = lambda_param * relevance - (1 - lambda_param) * diversity
            mmr_scores.append((score, idx))

        best = max(mmr_scores, key=lambda x: x[0])[1]

        selected.append(best)
        candidates.remove(best)

    return selected

In [ ]:
def retrieve_chunks(query, k=3):
    query_embedding = model.encode([query])[0]

    selected_indices = mmr(query_embedding, chunk_embeddings, k)

    results = []
    for i in selected_indices:
        results.append({
            "text": chunks[i]["text"],
            "page": chunks[i]["page"]
        })

    return results

In [ ]:
def ask_question(query):
    retrieved = retrieve_chunks(query, k=5)

    context = "\n\n".join(
        [f"(Page {r['page']}) {r['text']}" for r in retrieved]
    )

    prompt = f"""
    You are a precise assistant.

    Use ONLY the provided context.
    If multiple pieces of information exist, synthesize them.
    Cite page numbers clearly.
    If not found, say "Not found".

    Context:
    {context}

    Question:
    {query}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
test_questions = [
    "What is a data incident?",
    "Who published the guidance notes?",
    "What is the purpose of the document?"
]

In [ ]:
for q in test_questions:
    print("\nQUESTION:", q)
    print("ANSWER:", ask_question(q))

In [ ]:
!pip install rank-bm25

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [c["text"].split() for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

In [ ]:
def hybrid_retrieve(query, k=5):
    # Semantic scores
    query_embedding = model.encode([query])[0]
    semantic_scores = np.dot(chunk_embeddings, query_embedding)

    # BM25 scores
    tokenized_query = query.split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # Normalize scores
    semantic_scores = (semantic_scores - np.min(semantic_scores)) / (np.max(semantic_scores) - np.min(semantic_scores) + 1e-8)
    bm25_scores = (bm25_scores - np.min(bm25_scores)) / (np.max(bm25_scores) - np.min(bm25_scores) + 1e-8)

    # Combine
    combined_scores = 0.6 * semantic_scores + 0.4 * bm25_scores

    top_indices = np.argsort(combined_scores)[-k:][::-1]

    results = []
    for i in top_indices:
        results.append({
            "text": chunks[i]["text"],
            "page": chunks[i]["page"]
        })

    return results

In [ ]:
hybrid_retrieve("DG ECHO support")

In [ ]:
def ask_question(query):
    retrieved = hybrid_retrieve(query, k=5)

    context = "\n\n".join(
        [f"(Page {r['page']}) {r['text']}" for r in retrieved]
    )

    prompt = f"""
    You are a strict factual assistant.

    RULES:
    - Use ONLY the provided context
    - Do NOT add outside knowledge
    - If answer is incomplete, say "Partially found"
    - If not found, say "Not found"
    - Always cite page numbers

    Context:
    {context}

    Question:
    {query}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
def reset_pipeline():
    global chunks, chunk_embeddings, index, bm25

    chunks = []
    chunk_embeddings = None
    index = None
    bm25 = None

    print("Pipeline reset complete.")

In [ ]:
from google.colab import files
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

In [ ]:
# 1. Reset
reset_pipeline()

# 2. Extract
pages = extract_text_with_pages(pdf_path)

# 3. Chunk
chunks = semantic_chunking_with_metadata(pages)

# 4. Embeddings
texts = [c["text"] for c in chunks]
chunk_embeddings = model.encode(texts)

# 5. FAISS
index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
index.add(np.array(chunk_embeddings))

# 6. BM25
from rank_bm25 import BM25Okapi
tokenized_chunks = [c["text"].split() for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

In [ ]:
ask_question("What is a data incident?")